# Ch 13 · Time Integration: Euler, RK4, Implicit and Discrete

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/main/dpdk-research/mujoco-warp-course/notebooks/13-integrators.ipynb)

Companion notebook for the chapter page `chapters/13-integrators.html` ([page](../chapters/13-integrators.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

Every code cell below is the same file as the matching block on the chapter page (`examples/13-integrators/`). Budgets grow automatically on a GPU (`ON_GPU`); on a CPU they stay small. Timings printed here are **your** numbers, and CPU numbers are never GPU numbers.

The examples run in numeric order. The page shows them where each concept is taught.

## 1. One step = forward + integrate
`step` runs `forward`, then dispatches on the Python int `m.opt.integrator`: `euler`, `rungekutta4`, `implicit` (IMPLICIT and IMPLICITFAST) or `discrete`. All of them finish in `_advance`: activation, **velocity first**, then position with the new velocity (semi-implicit), plastic updates, history, time + overflow bits, warm start.

Two ways to choose the integrator: MJCF `<option integrator=...>` or `m.opt.integrator = mjw.IntegratorType.X`. After switching, **recapture any CUDA graph**.

In [ ]:
# Example 13.1 - ex1_switch.py
"""Example 13.1 - choose the integrator in MJCF or on the device model.

Two ways to pick an integrator: the MJCF attribute <option integrator="..."/> (read by put_model),
or the Python int m.opt.integrator = mjw.IntegratorType.X on an existing mjw.Model. Both give
the same trajectory. The double pendulum starts at q = (1.2, 0) and runs 100 steps per integrator.
Run: python ex1_switch.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=5, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01" integrator="{integrator}"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 2
NSTEP = 100
Q0 = np.array([1.2, 0.0])

# MJCF spelling -> enum member (the XML attribute is case-sensitive)
NAMES = {"Euler": mjw.IntegratorType.EULER, "RK4": mjw.IntegratorType.RK4,
         "implicit": mjw.IntegratorType.IMPLICIT, "implicitfast": mjw.IntegratorType.IMPLICITFAST,
         "discrete": mjw.IntegratorType.DISCRETE}


def run(m, mjm):
    d = mjw.make_data(mjm, nworld=NWORLD)
    d.qpos.assign(np.tile(Q0, (NWORLD, 1)))
    for _ in range(NSTEP):
        mjw.step(m, d)
    return d.qpos.numpy()[0]


# way B: one model, switch the Python int on the device model
mjm_b = mujoco.MjModel.from_xml_string(PENDULUM.format(integrator="Euler"))
m_b = mjw.put_model(mjm_b)
print("type(m.opt.integrator):", type(m_b.opt.integrator).__name__, "| value:", m_b.opt.integrator)

print(f"{'integrator':<13}{'enum':>5}  {'qpos via MJCF':<22}{'qpos via m.opt':<22}{'max diff':>9}")
for name, enum in NAMES.items():
    # way A: MJCF attribute, compiled by MuJoCo, copied by put_model
    mjm_a = mujoco.MjModel.from_xml_string(PENDULUM.format(integrator=name))
    q_a = run(mjw.put_model(mjm_a), mjm_a)
    m_b.opt.integrator = enum          # a plain int: read when step() dispatches
    q_b = run(m_b, mjm_b)
    print(f"{name:<13}{int(enum):>5}  {np.array2string(q_a):<22}{np.array2string(q_b):<22}"
          f"{np.abs(q_a - q_b).max():>9.1e}")

## 2. Parity with MuJoCo C
Each integrator should follow `mujoco.mj_step` with the same integrator to float32 accuracy (about 1e-6), with and without damping.

In [ ]:
# Example 13.2 - ex2_parity.py
"""Example 13.2 - every integrator against MuJoCo C (mujoco.mj_step).

The double pendulum from q = (1.2, 0) runs 300 steps (dt = 0.01) in MJWarp and in C MuJoCo with
the same integrator, without and with joint damping 5. The table shows the largest |delta qpos|
over the whole trajectory. MJWarp computes in float32, C MuJoCo in float64.
Run: python ex2_parity.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 1           # one world is enough to compare with C; the code is the same for 4096
NSTEP = 300
INTEGRATORS = [("Euler", mujoco.mjtIntegrator.mjINT_EULER),
               ("RK4", mujoco.mjtIntegrator.mjINT_RK4),
               ("implicit", mujoco.mjtIntegrator.mjINT_IMPLICIT),
               ("implicitfast", mujoco.mjtIntegrator.mjINT_IMPLICITFAST),
               ("discrete", mujoco.mjtIntegrator.mjINT_DISCRETE)]


def max_diff(integrator, damping):
    mjm = mujoco.MjModel.from_xml_string(PENDULUM)
    mjm.opt.integrator = integrator
    mjm.dof_damping[:] = damping
    mjd = mujoco.MjData(mjm)
    mjd.qpos[:] = [1.2, 0.0]
    m = mjw.put_model(mjm)
    d = mjw.put_data(mjm, mjd, nworld=NWORLD)
    worst = 0.0
    for _ in range(NSTEP):
        mujoco.mj_step(mjm, mjd)
        mjw.step(m, d)
        worst = max(worst, float(np.abs(d.qpos.numpy()[0] - mjd.qpos).max()))
    return worst, mjd.qpos.copy()


print(f"{'integrator':<13}{'damping 0':>11}{'damping 5':>11}   C qpos after 3 s (damping 5)")
for name, integ in INTEGRATORS:
    e0, _ = max_diff(integ, 0.0)
    e5, q5 = max_diff(integ, 5.0)
    print(f"{name:<13}{e0:>11.1e}{e5:>11.1e}   {np.array2string(q5, precision=4)}")

## 3. The energy race
Undamped double pendulum, q = (1.2, 0), dt = 0.02, 6 s. Euler gains energy, IMPLICIT loses some (implicit treatment of Coriolis forces is dissipative), RK4 holds it, IMPLICITFAST equals Euler because no smooth force depends on velocity (D = 0). The example calls `mjw.forward` before each sample because RK4 leaves `d.energy` from its last stage.

In [ ]:
# Example 13.3 - ex3_energy_race.py
"""Example 13.3 - the energy race: Euler, RK4, implicit and implicitfast on an undamped pendulum.

The double pendulum has no damping and no control, so its total energy should stay constant.
Each integrator runs 6 s at dt = 0.02 from q = (1.2, 0). Energy is the [potential, kinetic] pair
in d.energy (the model enables <flag energy="enable"/>). We call mjw.forward before each sample so
every integrator reports the energy of its current state (see Example 13.6 for why that matters
under RK4).
Run: python ex3_energy_race.py [--dump-json PATH]
  --dump-json PATH writes energy curves and decimated joint angles for the page's chart and replay.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 1           # every world would be identical; the integrators run as separate models
DT, T_END = 0.02, 6.0
NSTEP = int(round(T_END / DT))   # 300
EVERY = 5                        # sample every 5 steps (0.1 s) for the chart and replay
INTEGRATORS = {"Euler": mjw.IntegratorType.EULER, "RK4": mjw.IntegratorType.RK4,
               "implicit": mjw.IntegratorType.IMPLICIT,
               "implicitfast": mjw.IntegratorType.IMPLICITFAST}

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
mjm.opt.timestep = DT
energy, angles = {}, {}
for name, integ in INTEGRATORS.items():
    m = mjw.put_model(mjm)
    m.opt.integrator = integ
    d = mjw.make_data(mjm, nworld=NWORLD)
    d.qpos.assign(np.array([[1.2, 0.0]] * NWORLD))
    e_hist, q_hist = [], []
    for i in range(NSTEP + 1):
        if i % EVERY == 0:
            mjw.forward(m, d)                     # energy of the current state
            e_hist.append(float(d.energy.numpy()[0].sum()))
            q_hist.append([round(float(v), 3) for v in d.qpos.numpy()[0]])
        if i < NSTEP:
            mjw.step(m, d)
    energy[name], angles[name] = e_hist, q_hist

print(f"total energy [J] every 0.6 s, dt = {DT}, {NSTEP} steps")
for name, e in energy.items():
    print(f"{name:<13}", np.array2string(np.array(e[::6]), precision=3, max_line_width=200))
print(f"{'integrator':<13}{'E(0)':>8}{'E(6 s)':>9}{'change':>9}")
for name, e in energy.items():
    print(f"{name:<13}{e[0]:>8.3f}{e[-1]:>9.3f}{100 * (e[-1] - e[0]) / e[0]:>8.1f}%")

if args.dump_json:
    names = list(INTEGRATORS)
    out = {"nworld": len(names), "dt": DT * EVERY, "kind": "pendulum2d", "labels": names,
           "energy": {k: [round(v, 4) for v in e] for k, e in energy.items()},
           "frames": [[angles[k][f] for k in names] for f in range(len(angles[names[0]]))]}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(out["frames"]), "frames")

In [ ]:
# Interactive energy plot: uses `energy` and DT/EVERY from Example 13.3 (run it first).
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

def plot_energy(Euler=True, RK4=True, implicit=True, implicitfast=False):
    show = dict(Euler=Euler, RK4=RK4, implicit=implicit, implicitfast=implicitfast)
    t = np.arange(len(energy["Euler"])) * DT * EVERY
    plt.figure(figsize=(7, 3.5))
    for name, e in energy.items():
        if show[name]:
            plt.plot(t, e, label=name, ls="--" if name == "implicitfast" else "-")
    plt.xlabel("time [s]"); plt.ylabel("total energy [J]"); plt.legend(); plt.grid(alpha=.3)
    plt.show()

try:
    from ipywidgets import interact
    interact(plot_energy)
except ImportError:
    plot_energy()

## 4. Implicit damping
Euler solves `(M + h B) qacc = M qacc_forward` unless EULERDAMP (or DAMPER) is disabled. Explicit damping multiplies v by `1 - h b/m` per step (unstable above 2); implicit damping by `1 / (1 + h b/m)` (always stable). Eight worlds, eight dampings, one launch. The last line shows a 4.0.1 pitfall: with `njmax=0`, `d.efc.Ma` is never filled and the damping solve returns zero acceleration.

In [ ]:
# Example 13.4 - ex4_damping_stability.py
"""Example 13.4 - implicit damping: why Euler survives very stiff damping.

A 1 kg slider with joint damping b and no gravity starts at qvel = 1. With step h, the explicit
update v <- v - h*b/m*v multiplies v by (1 - h*b/m) per step and blows up once h*b/m > 2.
MuJoCo's Euler integrator treats joint damping implicitly: it solves (M + h*D) qacc = f, which
multiplies v by 1 / (1 + h*b/m) and is stable for every b. Eight worlds get eight dampings
(per-world m.dof_damping), so h*b/m sweeps 0.5 ... 4 in one launch.
Run: python ex4_damping_stability.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=3, suppress=False, linewidth=110)

SLIDER = """
<mujoco model="damped_slider">
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody>
    <body name="block">
      <joint name="slide" type="slide" axis="1 0 0" damping="100"/>
      <geom type="box" size=".1 .1 .1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NSTEP = 10
RATIO = np.array([0.5, 1.0, 1.5, 1.9, 2.1, 2.5, 3.0, 4.0])   # h*b/m, one per world
NWORLD = len(RATIO)

mjm = mujoco.MjModel.from_xml_string(SLIDER)
h, mass = mjm.opt.timestep, mjm.body_mass[1]


def run(disable_eulerdamp, njmax=None):
    m = mjw.put_model(mjm)
    if disable_eulerdamp:
        m.opt.disableflags |= mjw.DisableBit.EULERDAMP     # same as <flag eulerdamp="disable"/>
    # per-world damping: kernels read dof_damping[worldid % dof_damping.shape[0]]
    m.dof_damping = wp.array((RATIO * mass / h).reshape(NWORLD, 1), dtype=float)
    kw = {} if njmax is None else {"njmax": njmax}
    d = mjw.make_data(mjm, nworld=NWORLD, **kw)
    d.qvel.fill_(1.0)
    hist = [np.ones(NWORLD)]
    for _ in range(NSTEP):
        mjw.step(m, d)
        hist.append(d.qvel.numpy()[:, 0].copy())
    return np.array(hist)


implicit = run(disable_eulerdamp=False)
explicit = run(disable_eulerdamp=True)
print("h*b/m              ", RATIO)
print(f"|qvel| after {NSTEP} steps")
print("  Euler (default)  ", np.abs(implicit[-1]))
print("  predicted        ", (1 / (1 + RATIO)) ** NSTEP)
print("  EULERDAMP off    ", np.abs(explicit[-1]))
print("  predicted        ", np.abs(1 - RATIO) ** NSTEP)
print("unstable worlds (EULERDAMP off):", np.flatnonzero(np.abs(explicit[-1]) > 1).tolist())

# 4.0.1 pitfall: with njmax=0 the solver skips the code that fills d.efc.Ma, which the Euler
# damping solve uses as its right-hand side, so qacc = 0 and the damping does nothing.
frozen = run(disable_eulerdamp=False, njmax=0)
print("  Euler, njmax=0   ", np.abs(frozen[-1]), "<- 4.0.1 wheel: velocities frozen")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"ratio": RATIO.tolist(), "nstep": NSTEP,
                   "implicit": [[float(f"{v:.4e}") for v in row] for row in implicit.T],
                   "explicit": [[float(f"{v:.4e}") for v in row] for row in explicit.T]}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 5. What a step costs
Humanoid, all four standard integrators, after a warm-up step. RK4 costs about four forwards. The last lines compute the scratch memory IMPLICIT's RNE derivative allocates per step.

In [ ]:
# Example 13.5 - ex5_cost_table.py
"""Example 13.5 - what one step costs with each integrator (humanoid).

Times NSTEP steps of the shipped humanoid (nv 27) per integrator after a warm-up step (JIT), with
wp.synchronize() before reading the clock. On a CPU dev box only the ratios mean something; on a
Colab GPU the notebook prints your own numbers (without CUDA graph capture, so per-step allocations
inside the integrators are included; Ch 24 shows the captured version). The last lines compute the
scratch memory that IMPLICIT's RNE derivative allocates on every step.
Run: python ex5_cost_table.py [--dump-json PATH]
"""
import argparse
import json
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 200 if ON_GPU else 10

HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
mjm = mujoco.MjModel.from_xml_path(str(HUMANOID_PATH))
print(f"humanoid: nq {mjm.nq}, nv {mjm.nv}, nbody {mjm.nbody}, nu {mjm.nu} | nworld {NWORLD}, "
      f"device {wp.get_device()}")

INTEGRATORS = {"Euler": mjw.IntegratorType.EULER, "RK4": mjw.IntegratorType.RK4,
               "implicit": mjw.IntegratorType.IMPLICIT,
               "implicitfast": mjw.IntegratorType.IMPLICITFAST}
rng = np.random.default_rng(0)
times = {}
for name, integ in INTEGRATORS.items():
    m = mjw.put_model(mjm)
    m.opt.integrator = integ
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=32, njmax=128)
    d.ctrl.assign(rng.uniform(-0.2, 0.2, (NWORLD, mjm.nu)))
    mjw.step(m, d)                 # warm-up: compiles (or loads) this integrator's kernels
    wp.synchronize()
    t0 = time.perf_counter()
    for _ in range(NSTEP):
        mjw.step(m, d)
    wp.synchronize()
    times[name] = (time.perf_counter() - t0) / NSTEP
    assert not d.overflow.numpy().any(), "overflow: raise nconmax/njmax"

label = "GPU" if ON_GPU else "CPU dev box"
print(f"{'integrator':<13}{'ms/step':>9}{'x Euler':>9}{'world-steps/s':>15}   ({label})")
for name, t in times.items():
    print(f"{name:<13}{1e3 * t:>9.2f}{t / times['Euler']:>9.2f}{NWORLD / t:>15,.0f}")
print("overflow: none")

# deriv_rne_vel allocates Dcvel, Dcacc, Dcfrcbody (nworld, nbody, nv) and Dcdof_dot (nworld, nv, nv)
# of wp.spatial_vector (6 float32 = 24 bytes) on every IMPLICIT step
for nw in (NWORLD, 4096, 65536):
    nbytes = nw * (3 * mjm.nbody * mjm.nv + mjm.nv * mjm.nv) * 24
    print(f"IMPLICIT RNE-derivative scratch at nworld={nw:>6}: {nbytes / 2**20:9.1f} MiB per step")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": label, "nworld": NWORLD, "nstep": NSTEP,
                   "ms": {k: round(1e3 * v, 3) for k, v in times.items()}}, f)
    print("wrote", args.dump_json)

## 6. RK4 and sensors
Stages 2-4 call `_forward(skip_sensor=True)`, so `sensordata` keeps the start-of-step values; `energy` comes from the last stage. C MuJoCo agrees.

In [ ]:
# Example 13.6 - ex6_rk4_sensors.py
"""Example 13.6 - under RK4, sensordata holds the values from the start of the step.

rungekutta4 runs _forward three more times at perturbed states with skip_sensor=True, so the
sensors written by the first forward (state at time t) survive. The double pendulum's jointpos
sensor q1 therefore equals qpos[0] *before* the step, not after it. d.energy is different: it is
recomputed in every stage, so after an RK4 step it describes the last perturbed stage (close
to the end-of-step state). C MuJoCo
(mujoco.mj_step) is printed next to MJWarp for both.
Run: python ex6_rk4_sensors.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=5, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 2

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
mjm.opt.integrator = mujoco.mjtIntegrator.mjINT_RK4
mjd = mujoco.MjData(mjm)
mujoco.mj_resetDataKeyframe(mjm, mjd, 0)          # q = (pi/2, 0): horizontal, at rest
mujoco.mj_step(mjm, mjd)                          # one step so qvel is not zero
m = mjw.put_model(mjm)
d = mjw.put_data(mjm, mjd, nworld=NWORLD)

for _ in range(20):
    mujoco.mj_step(mjm, mjd)
    mjw.step(m, d)

q_before, v_before = d.qpos.numpy()[0].copy(), d.qvel.numpy()[0].copy()
mjw.forward(m, d)                                 # no integration: energy of the start state
e_start = d.energy.numpy()[0].copy()
mujoco.mj_step(mjm, mjd)
mjw.step(m, d)
sens = d.sensordata.numpy()[0]
print(f"{'quantity':<22}{'before step':>13}{'after step':>13}{'sensordata':>13}{'C sensordata':>14}")
print(f"{'q1 (jointpos)':<22}{q_before[0]:>13.5f}{d.qpos.numpy()[0, 0]:>13.5f}{sens[0]:>13.5f}"
      f"{mjd.sensordata[0]:>14.5f}")
print(f"{'v1 (jointvel)':<22}{v_before[0]:>13.5f}{d.qvel.numpy()[0, 0]:>13.5f}{sens[1]:>13.5f}"
      f"{mjd.sensordata[1]:>14.5f}")
print("sensordata == start-of-step state:",
      bool(np.allclose(sens[:2], [q_before[0], v_before[0]], atol=1e-6)))

# energy of the state after the step: recompute with forward (no integration)
e_step = d.energy.numpy()[0].copy()
mjw.forward(m, d)
print("energy [pot, kin] of the start state:", e_start)
print("d.energy right after the RK4 step  :", e_step, "| C:", mjd.energy)
print("energy of the post-step state      :", d.energy.numpy()[0], "(mjw.forward)")

## 7. The IMPLICITFAST matrix
`mjw.deriv_smooth_vel(m, d, out)` writes `M - dt*D` in M's sparse layout (`m.nC` values per world). Densify with `mujoco.mju_sym2dense` and compare with a float64 finite-difference reference.

In [ ]:
# Example 13.7 - ex7_deriv_smooth_vel.py
"""Example 13.7 - the matrix IMPLICITFAST factors: M - dt * D, from mjw.deriv_smooth_vel.

D = d(qfrc_passive + qfrc_actuator)/d(qvel) is the velocity derivative of the smooth forces.
The pendulum gets joint damping (1.0, 0.5) and a velocity servo (kv = 3) on hinge2, so D is not
zero. mjw.deriv_smooth_vel writes M - dt*D in MuJoCo's sparse lower-triangular layout of M
(m.nC values per world, the way forward.py allocates it). We densify it with mujoco.mju_sym2dense
and compare it with M (mujoco.mj_fullM) minus dt times a central finite-difference D from C MuJoCo.
Run: python ex7_deriv_smooth_vel.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=5, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 2

# damped variant of the shared model: joint damping and a velocity servo on the elbow
DAMPED = (PENDULUM
          .replace('name="hinge1" type="hinge"', 'name="hinge1" type="hinge" damping="1.0"')
          .replace('name="hinge2" type="hinge"', 'name="hinge2" type="hinge" damping="0.5"')
          .replace('</actuator>', '<velocity name="elbow_kv" joint="hinge2" kv="3"/></actuator>')
          .replace('timestep="0.01"', 'timestep="0.01" integrator="implicitfast"'))
mjm = mujoco.MjModel.from_xml_string(DAMPED)
mjd = mujoco.MjData(mjm)
mjd.qpos[:] = [0.7, -0.4]
mjd.qvel[:] = [1.0, 2.0]
dt = mjm.opt.timestep

m = mjw.put_model(mjm)
d = mjw.put_data(mjm, mjd, nworld=NWORLD)
mjw.fwd_position(m, d, factorize=False)
mjw.fwd_velocity(m, d)
mjw.fwd_actuation(m, d)
out = wp.empty((d.nworld, m.nC), dtype=float)       # sparse layout of M: nC values per world
mjw.deriv_smooth_vel(m, d, out)

print(f"nv = {mjm.nv}, nC = {mjm.nC}, M_rownnz = {mjm.M_rownnz}, M_rowadr = {mjm.M_rowadr}, "
      f"M_colind = {mjm.M_colind}")
vals = out.numpy()[0].astype(np.float64)
print("sparse values of world 0:", vals)
H_mjw = np.zeros((mjm.nv, mjm.nv))
mujoco.mju_sym2dense(H_mjw, vals, mjm.M_rownnz, mjm.M_rowadr, mjm.M_colind)

# reference in float64: M from C MuJoCo (MuJoCo 4 has no mjd.qM; mj_fullM reads mjd.M)
mujoco.mj_forward(mjm, mjd)
M = np.zeros((mjm.nv, mjm.nv))
mujoco.mj_fullM(mjm, mjd, M)


def smooth_force(qvel):
    dd = mujoco.MjData(mjm)
    dd.qpos[:], dd.qvel[:], dd.ctrl[:] = mjd.qpos, qvel, mjd.ctrl
    mujoco.mj_forward(mjm, dd)
    return dd.qfrc_passive + dd.qfrc_actuator


eps = 1e-6
D = np.column_stack([(smooth_force(mjd.qvel + eps * e) - smooth_force(mjd.qvel - eps * e)) / (2 * eps)
                     for e in np.eye(mjm.nv)])
H_ref = M - dt * D
print("M (mj_fullM):\n", M)
print("D = d(qfrc_passive + qfrc_actuator)/dqvel (finite differences):\n", D)
print("M - dt*D from mjw.deriv_smooth_vel:\n", H_mjw)
print("M - dt*D reference:\n", H_ref)
print(f"max |difference| = {np.abs(H_mjw - H_ref).max():.1e}")

## 8. step2 with RK4 runs Euler
`step2` ends with `# note: RK4 defaults to Euler`. Keep RK4 and use `m.callback.control` if you need code inside the step. Under RK4 the callback runs at each of the 4 stage states and stages 2-4 skip sensors, so keep it a cheap, side-effect-free function of the state.

In [ ]:
# Example 13.8 - ex8_step2_rk4.py
"""Example 13.8 - step1/step2 with an RK4 model integrates with Euler.

step2 ends with "integrate with Euler, implicitfast, or discrete" (forward.py): for RK4 it falls
back to euler(). Three copies of the double pendulum run 100 steps from q = (1.2, 0):
step() with RK4, step1()+step2() with the same RK4 model, and step() with an Euler model.
Run: python ex8_step2_rk4.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=6, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 2
NSTEP = 100

mjm = mujoco.MjModel.from_xml_string(PENDULUM)


def make(integrator):
    m = mjw.put_model(mjm)
    m.opt.integrator = integrator
    d = mjw.make_data(mjm, nworld=NWORLD)
    d.qpos.assign(np.tile([1.2, 0.0], (NWORLD, 1)))
    return m, d


m_rk, d_step = make(mjw.IntegratorType.RK4)
_, d_split = make(mjw.IntegratorType.RK4)
m_eu, d_euler = make(mjw.IntegratorType.EULER)

for _ in range(NSTEP):
    mjw.step(m_rk, d_step)
    mjw.step1(m_rk, d_split)
    # ... a policy would read d_split.sensordata and write d_split.ctrl here ...
    mjw.step2(m_rk, d_split)
    mjw.step(m_eu, d_euler)

q_step, q_split, q_euler = (x.qpos.numpy()[0] for x in (d_step, d_split, d_euler))
print("RK4 model, step()        :", q_step)
print("RK4 model, step1 + step2 :", q_split)
print("Euler model, step()      :", q_euler)
print(f"|split - RK4 step|   = {np.abs(q_split - q_step).max():.2e}")
print(f"|split - Euler step| = {np.abs(q_split - q_euler).max():.2e}  <- step2 ran Euler")

## 9. Convergence study in one launch
`m.opt.timestep` is batched: one timestep per world. Errors at t = 1 s against a float64 RK4 reference give the order of each method, and the float32 floor of RK4. This cell takes about a minute on CPU.

In [ ]:
# Example 13.9 - ex9_dt_sweep.py
"""Example 13.9 - a convergence study in one launch: one timestep per world.

m.opt.timestep is a batched field: kernels read opt_timestep[worldid % opt_timestep.shape[0]].
Eight worlds get eight timesteps (0.1 ... 0.0005 s) and all of them step together. World k reaches
T = 1 s after T/dt_k steps; we read its qpos at exactly that step and compare it with a fine
float64 reference (C MuJoCo, RK4, dt = 1e-4). The slope of log(error) against log(dt) is the
order of the method: about 1 for Euler, about 4 for RK4 (until float32 round-off takes over).
Run: python ex9_dt_sweep.py [--dump-json PATH] [--dump-png DIR]
"""
import argparse
import json
import os

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
T_END = 1.0
DTS = np.array([0.1, 0.05, 0.025, 0.0125, 0.005, 0.0025, 0.001, 0.0005])   # one world per dt
NWORLD = len(DTS)
NSTEPS = np.rint(T_END / DTS).astype(int)                                 # 10 ... 2000
Q0 = [1.2, 0.0]
INTEGRATORS = {"Euler": mjw.IntegratorType.EULER, "RK4": mjw.IntegratorType.RK4,
               "implicit": mjw.IntegratorType.IMPLICIT,
               "implicitfast": mjw.IntegratorType.IMPLICITFAST}

mjm = mujoco.MjModel.from_xml_string(PENDULUM)

# float64 reference: C MuJoCo, RK4, tiny step
ref = mujoco.MjModel.from_xml_string(PENDULUM)
ref.opt.integrator, ref.opt.timestep = mujoco.mjtIntegrator.mjINT_RK4, 1e-4
rd = mujoco.MjData(ref)
rd.qpos[:] = Q0
for _ in range(int(round(T_END / 1e-4))):
    mujoco.mj_step(ref, rd)
q_ref = rd.qpos.copy()
print("reference qpos(T=1 s):", np.round(q_ref, 6))

errors = {}
for name, integ in INTEGRATORS.items():
    m = mjw.put_model(mjm)
    m.opt.integrator = integ
    m.opt.timestep = wp.array(DTS, dtype=float)       # per-world timestep, shape (nworld,)
    d = mjw.make_data(mjm, nworld=NWORLD)
    d.qpos.assign(np.tile(Q0, (NWORLD, 1)))
    q_at_T = np.zeros((NWORLD, mjm.nq))
    for i in range(1, NSTEPS.max() + 1):
        mjw.step(m, d)
        for w in np.flatnonzero(NSTEPS == i):          # world w just reached T
            q_at_T[w] = d.qpos.numpy()[w]
            assert abs(d.time.numpy()[w] - T_END) < 1e-4
    errors[name] = np.abs(q_at_T - q_ref).max(axis=1)


def order(err, dts, floor=2e-5):
    """Least-squares slope of log(err) vs log(dt), using points above the float32 floor."""
    keep = err > floor
    return float(np.polyfit(np.log(dts[keep]), np.log(err[keep]), 1)[0]), int(keep.sum())


print("dt [s]       " + "".join(f"{dt:>10.4f}" for dt in DTS))
for name, err in errors.items():
    print(f"{name:<13}" + "".join(f"{e:>10.1e}" for e in err))
print("estimated order (slope of log error vs log dt):")
for name, err in errors.items():
    p, n = order(err, DTS)
    print(f"  {name:<13}{p:5.2f}  (fit over {n} timesteps)")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"dts": DTS.tolist(), "t_end": T_END,
                   "errors": {k: [float(f"{e:.3e}") for e in v] for k, v in errors.items()}}, f)
    print("wrote", args.dump_json)

if not args.dump_png:                                # write files only when asked
    print("plot not saved (pass --dump-png DIR to write ex9_dt_sweep.png)")
else:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("skipped plot: matplotlib not installed")
    else:
        fig, ax = plt.subplots(figsize=(5, 4))
        for name, err in errors.items():
            ax.loglog(DTS, err, "o-", label=name)
        ax.loglog(DTS, 3 * DTS, "k:", label="slope 1")
        ax.loglog(DTS, 0.3 * DTS ** 4, "k--", label="slope 4")
        ax.set_xlabel("timestep dt [s]")
        ax.set_ylabel("max |qpos - reference| at t = 1 s")
        ax.legend()
        path = os.path.join(args.dump_png, "ex9_dt_sweep.png")
        fig.savefig(path, dpi=80, bbox_inches="tight")
        print("saved", path)

In [ ]:
# Log-log plot of the errors left behind by Example 13.9 (DTS and errors).
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
for name, err in errors.items():
    ax.loglog(DTS, err, marker="o", label=name, ls="--" if name == "implicitfast" else "-")
ax.loglog(DTS, 3 * DTS, "k:", label="slope 1")
ax.loglog(DTS, 0.3 * DTS ** 4, "k--", label="slope 4")
ax.set_xlabel("timestep dt [s]"); ax.set_ylabel("max |qpos - reference| at t = 1 s")
ax.legend(); plt.show()

## Exercise 13.1: measure the order of RK4
Uses `DTS` and `errors` from Example 13.9. Fit the slope of log(error) against log(dt) above the float32 floor.

In [ ]:
# Needs DTS and errors from Example 13.9 (run that cell first).
import numpy as np


def fit_order(dts, err, floor=2e-5):
    keep = ...            # drop the points at the float32 floor
    return ...            # least-squares slope of log(err) against log(dt)


p_rk4, p_euler = fit_order(DTS, errors["RK4"]), fit_order(DTS, errors["Euler"])
print(f"order: RK4 {p_rk4:.2f}, Euler {p_euler:.2f}")
assert p_rk4 > 3
assert 0.8 < p_euler < 1.3

In [ ]:
#@title Exercise 13.1: solution (double-click to show the code)
def fit_order(dts, err, floor=2e-5):
    keep = err > floor
    return float(np.polyfit(np.log(dts[keep]), np.log(err[keep]), 1)[0])


p_rk4, p_euler = fit_order(DTS, errors["RK4"]), fit_order(DTS, errors["Euler"])
print(f"order: RK4 {p_rk4:.2f}, Euler {p_euler:.2f}")
assert p_rk4 > 3
assert 0.8 < p_euler < 1.3

## 10. Quaternion integration
`_next_position` rotates free and ball joints with `math.quat_integrate` (an exact rotation per step). The naive add-and-renormalize update drifts with h^3 per step.

In [ ]:
# Example 13.10 - ex10_quat_integrate.py
"""Example 13.10 - integrating orientation: the exact quaternion update vs "add and renormalize".

A free cube spins with gravity off. A cube has equal principal inertias, so its body-frame angular
velocity qvel[3:6] stays constant and the exact answer is known: q(t) = q0 * exp(omega * t / 2).
forward.py::_next_position calls math.quat_integrate for free and ball joints, which multiplies
by the exact rotation of one step. The naive update q <- normalize(q + dt/2 * q * (0, omega))
(NumPy, float64) drifts. Five worlds use five timesteps; every world runs 500 steps.
Run: python ex10_quat_integrate.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=5, suppress=True)

CUBE = """
<mujoco model="spinning_cube">
  <option timestep="0.01" gravity="0 0 0"/>
  <worldbody>
    <body name="cube" pos="0 0 1">
      <freejoint/>
      <geom type="box" size=".1 .1 .1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
# Q-prefixed names: in the notebook this cell must not overwrite DTS / EVERY from Examples 13.3 and 13.9.
QDTS = np.array([0.002, 0.005, 0.01, 0.02, 0.05])
QNWORLD = len(QDTS)
QNSTEP = 500
QEVERY = 10
OMEGA = np.array([1.0, 2.0, 3.0])           # body-frame angular velocity [rad/s]


def qmul(a, b):
    w1, x1, y1, z1 = a
    w2, x2, y2, z2 = b
    return np.array([w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2, w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
                     w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2, w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2])


def qexp(omega, t):                          # rotation by |omega| t about omega
    th = np.linalg.norm(omega) * t
    return np.concatenate([[np.cos(th / 2)], np.sin(th / 2) * omega / np.linalg.norm(omega)])


def angle(a, b):                             # angle of the rotation a^-1 * b (robust near 0)
    r = qmul(a * [1, -1, -1, -1], b)
    return 2 * np.arctan2(np.linalg.norm(r[1:]), abs(r[0]))


mjm = mujoco.MjModel.from_xml_string(CUBE)
m = mjw.put_model(mjm)
m.opt.timestep = wp.array(QDTS, dtype=float)
d = mjw.make_data(mjm, nworld=QNWORLD)
qvel = np.zeros((QNWORLD, mjm.nv))
qvel[:, 3:6] = OMEGA
d.qvel.assign(qvel)
print("free joint: qpos = [x y z | qw qx qy qz] (7), qvel = [vx vy vz | wx wy wz] (6)")
print("qpos[0] at t=0:", d.qpos.numpy()[0], "| qvel[0]:", d.qvel.numpy()[0])

q0 = mjm.qpos0[3:7].copy()
naive = np.tile(q0, (QNWORLD, 1))
norm_before = np.zeros(QNWORLD)
rec = {"mjw": [], "naive": [], "exact": [], "err_mjw": [], "err_naive": []}
for i in range(QNSTEP + 1):
    if i % QEVERY == 0:
        q_mjw = d.qpos.numpy()[:, 3:7].astype(np.float64)
        exact = np.array([qmul(q0, qexp(OMEGA, i * dt)) for dt in QDTS])
        rec["mjw"].append(np.round(q_mjw, 4).tolist())
        rec["naive"].append(np.round(naive, 4).tolist())
        rec["exact"].append(np.round(exact, 4).tolist())
        rec["err_mjw"].append([angle(a, b) for a, b in zip(q_mjw, exact)])
        rec["err_naive"].append([angle(a, b) for a, b in zip(naive, exact)])
    if i == QNSTEP:
        break
    mjw.step(m, d)
    for w, dt in enumerate(QDTS):                 # naive: q += dt/2 * q * (0, omega), renormalize
        qn = naive[w] + 0.5 * dt * qmul(naive[w], np.concatenate([[0.0], OMEGA]))
        norm_before[w] = np.linalg.norm(qn)
        naive[w] = qn / norm_before[w]

print(f"after {QNSTEP} steps, |omega| = {np.linalg.norm(OMEGA):.3f} rad/s")
print(f"{'dt [s]':>8}{'turned [rad]':>14}{'MJWarp err [deg]':>18}{'naive err [deg]':>17}"
      f"{'|q| before renorm':>19}")
for w, dt in enumerate(QDTS):
    print(f"{dt:>8.3f}{np.linalg.norm(OMEGA) * dt * QNSTEP:>14.2f}"
          f"{np.degrees(rec['err_mjw'][-1][w]):>18.1e}{np.degrees(rec['err_naive'][-1][w]):>17.4f}"
          f"{norm_before[w]:>19.6f}")
print("MJWarp |q| after the run:", np.linalg.norm(d.qpos.numpy()[:, 3:7], axis=1))

if args.dump_json:
    rec["err_mjw"] = [[float(f"{v:.2e}") for v in r] for r in np.degrees(rec["err_mjw"])]
    rec["err_naive"] = [[float(f"{v:.3e}") for v in r] for r in np.degrees(rec["err_naive"])]
    with open(args.dump_json, "w") as f:
        json.dump({"dts": QDTS.tolist(), "omega": OMEGA.tolist(), "every": QEVERY, "nstep": QNSTEP,
                   **rec}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## Exercise 13.2: largest stable timestep per integrator
A very stiff damper on the cartpole's cart; twelve timesteps in one launch per case. Predict first: explicit damping is stable for h < 2m/b with m = 1.1 kg; RK4's real stability interval ends near -2.785.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

CARTPOLE = """
<mujoco model="cartpole">
  <option timestep="0.01" integrator="implicitfast"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="front" pos="0 -4 1" xyaxes="1 0 0 0 0 1"/>
    <geom name="rail" type="capsule" fromto="-2 0 1 2 0 1" size=".02" contype="0" conaffinity="0"/>
    <body name="cart" pos="0 0 1">
      <joint name="slider" type="slide" axis="1 0 0" range="-1.8 1.8" limited="true" damping="0.1"/>
      <geom name="cart" type="box" size=".2 .1 .05" mass="1" contype="0" conaffinity="0"/>
      <body name="pole">
        <joint name="hinge" type="hinge" axis="0 1 0" damping="0.01"/>
        <geom name="pole" type="capsule" fromto="0 0 0 0 0 .6" size=".03" mass="0.1"
              contype="0" conaffinity="0"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="push" joint="slider" gear="10" ctrlrange="-1 1" ctrllimited="true"/></actuator>
  <sensor>
    <jointpos joint="slider"/><jointpos joint="hinge"/>
    <jointvel joint="slider"/><jointvel joint="hinge"/>
  </sensor>
</mujoco>
"""
DTS = np.array([0.002, 0.004, 0.006, 0.008, 0.01, 0.012, 0.014, 0.016, 0.02, 0.025, 0.03, 0.04])
mjm = mujoco.MjModel.from_xml_string(CARTPOLE)
mjm.dof_damping[:] = [200.0, 0.01]          # a very stiff damper on the cart
CASES = {"Euler": (mjw.IntegratorType.EULER, False),
         "Euler, EULERDAMP off": (mjw.IntegratorType.EULER, True),
         "RK4": (mjw.IntegratorType.RK4, False),
         "implicitfast": (mjw.IntegratorType.IMPLICITFAST, False)}


def largest_stable_dt(integrator, no_eulerdamp, nstep=100):
    ...   # one model, per-world timesteps, qvel = (1.0, 0.5) in every world, nstep steps
    ...   # return the largest dt before the first unstable world


dt_max = {name: largest_stable_dt(*case) for name, case in CASES.items()}
for name, dt in dt_max.items():
    print(f"{name:<22} largest stable dt = {dt:.3f} s")
assert dt_max["implicitfast"] >= dt_max["RK4"] > dt_max["Euler, EULERDAMP off"]
assert dt_max["Euler"] > dt_max["Euler, EULERDAMP off"]

In [ ]:
#@title Exercise 13.2: solution (double-click to show the code)
def largest_stable_dt(integrator, no_eulerdamp, nstep=100):
    m = mjw.put_model(mjm)
    m.opt.integrator = integrator
    if no_eulerdamp:
        m.opt.disableflags |= mjw.DisableBit.EULERDAMP
    m.opt.timestep = wp.array(DTS, dtype=float)        # one timestep per world
    d = mjw.make_data(mjm, nworld=len(DTS))
    d.qvel.assign(np.tile([1.0, 0.5], (len(DTS), 1)))
    for _ in range(nstep):
        mjw.step(m, d)
    v = d.qvel.numpy()
    stable = np.isfinite(v).all(axis=1) & (np.abs(v).max(axis=1) < 10)
    first_bad = np.flatnonzero(~stable)
    return DTS[-1] if first_bad.size == 0 else DTS[: first_bad[0]].max()


dt_max = {name: largest_stable_dt(*case) for name, case in CASES.items()}
for name, dt in dt_max.items():
    print(f"{name:<22} largest stable dt = {dt:.3f} s")
assert dt_max["implicitfast"] >= dt_max["RK4"] > dt_max["Euler, EULERDAMP off"]
assert dt_max["Euler"] > dt_max["Euler, EULERDAMP off"]
print("ordering OK")

## Stretch: semi-implicit Euler for a free body, by hand
Take the cube of Example 13.10, switch gravity on and add a linear velocity. Each step: `mjw.forward`, read `d.qacc`, update `qvel` first, then the position with the new linear velocity and the quaternion with the exact exponential of the new body-frame angular velocity. Compare with `mjw.step` (EULERDAMP disabled or no damping).

In [ ]:
# your hand-written integrator here